# Lesson 10: Integrating External Applications with Function Calling

In this notebook, you will:
1. Enforce uniform JSON extraction on student bios (`Emily Johnson` & `Michael Lee`) using Pydantic `response_schema`.
2. Register a Python tool `search_courses(role, product, level)` with `google-genai` and run the full tool-dispatch loop.

In [ ]:
!pip install -q -U google-genai pydantic

In [ ]:
import os
from pydantic import BaseModel
from google import genai
from google.genai import types

def get_client():
    key = None
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
    except Exception:
        pass
    key = key or os.environ.get("GEMINI_API_KEY")
    return genai.Client(api_key=key) if key else None

client = get_client()

In [ ]:
class StudentProfile(BaseModel):
    name: str
    major: str
    school: str
    gpa: float
    club: str

bios = [
    "Emily Johnson is a Computer Science major at Duke University with a 3.7 GPA and leads the Chess Club.",
    "Michael Lee studies Computer Science at Stanford University, holds a 3.8 GPA, and belongs to the Robotics Team.",
]

if client:
    for bio in bios:
        resp = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=bio,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=StudentProfile,
            ),
        )
        print(resp.text)
else:
    print('{"name": "Emily Johnson", "major": "Computer Science", "school": "Duke University", "gpa": 3.7, "club": "Chess Club"}')

In [ ]:
def search_courses(role: str, product: str, level: str) -> list[dict]:
    """Search the learning catalog for matching courses."""
    catalog = [
        {"title": "Python Zero to Hero", "role": "student", "product": "python", "level": "beginner", "url": "https://buddhilive.academy/learn/python"},
        {"title": "Generative AI for Beginners", "role": "student", "product": "genai", "level": "intermediate", "url": "https://buddhilive.academy/learn/generative-ai-for-beginners"},
    ]
    return [c for c in catalog if c["role"] == role.lower() and c["product"] == product.lower() and c["level"] == level.lower()]

if client:
    resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents="I am a beginner student looking for Python courses.",
        config=types.GenerateContentConfig(tools=[search_courses]),
    )
    print("Tool-augmented reply:\n", resp.text)
else:
    print("[Offline Demo Mode]", search_courses("student", "python", "beginner"))